In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH   = '../data/raw/cell2celltrain.csv'
HOLDOUT_PATH = '../data/raw/cell2cellholdout.csv'
TARGET       = 'Churn'

In [2]:
df      = pd.read_csv(TRAIN_PATH)
holdout = pd.read_csv(HOLDOUT_PATH)

print(f"Train   : {df.shape}")
print(f"Holdout : {holdout.shape}")

Train   : (51047, 58)
Holdout : (20000, 58)


In [3]:
# Encode Churn: Yes→1, No→0
df[TARGET]      = df[TARGET].map({'Yes': 1, 'No': 0})
holdout[TARGET] = holdout[TARGET].map({'Yes': 1, 'No': 0})

print("Train churn rate  :", round(df[TARGET].mean(), 3))
print("Holdout churn rate:", round(holdout[TARGET].mean(), 3))

Train churn rate  : 0.288
Holdout churn rate: nan


In [4]:
# CustomerID is just an identifier — not a feature
drop_cols = ['CustomerID']
drop_cols = [c for c in drop_cols if c in df.columns]

df      = df.drop(columns=drop_cols)
holdout = holdout.drop(columns=drop_cols)
print(f"Dropped: {drop_cols}")

Dropped: ['CustomerID']


In [5]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Numeric → fill with median
for col in num_cols:
    median_val = df[col].median()
    df[col]      = df[col].fillna(median_val)
    holdout[col] = holdout[col].fillna(median_val)

# Categorical → fill with mode
for col in cat_cols:
    mode_val = df[col].mode()[0]
    df[col]      = df[col].fillna(mode_val)
    holdout[col] = holdout[col].fillna(mode_val)

print("Missing after fill:", df.isnull().sum().sum())

Missing after fill: 0


In [6]:
# Safe label encoding — handles unseen values in holdout
for col in cat_cols:
    # Get all unique values from BOTH train and holdout
    all_values = pd.concat([df[col], holdout[col]]).astype(str).unique()
    le = LabelEncoder()
    le.fit(all_values)  # fit on combined unique values
    
    df[col]      = le.transform(df[col].astype(str))
    holdout[col] = le.transform(holdout[col].astype(str))

print("All dtypes numeric:", (df.dtypes == 'object').sum() == 0)

All dtypes numeric: True


In [7]:
def build_features(data):
    df = data.copy()

    # 1. Revenue per month of tenure
    if 'MonthlyCharge' in df.columns and 'MonthsInService' in df.columns:
        df['revenue_per_month'] = df['MonthlyCharge'] / (df['MonthsInService'] + 1)

    # 2. Call quality ratio (dropped / total)
    if 'DroppedCalls' in df.columns and 'ReceivedCalls' in df.columns:
        total = df['ReceivedCalls'] + df['DroppedCalls'] + 1
        df['drop_call_ratio'] = df['DroppedCalls'] / total

    # 3. Average call duration
    if 'MinutesOfUse' in df.columns and 'MadeCallsCount' in df.columns:
        df['avg_call_duration'] = df['MinutesOfUse'] / (df['MadeCallsCount'] + 1)

    # 4. Overage flag (using more than plan allows)
    if 'OverageMinutes' in df.columns:
        df['has_overage'] = (df['OverageMinutes'] > 0).astype(int)

    # 5. Tenure band (loyalty tier)
    if 'MonthsInService' in df.columns:
        df['tenure_band'] = pd.cut(
            df['MonthsInService'],
            bins=[0, 12, 24, 48, 999],
            labels=[0, 1, 2, 3]
        ).astype(int)

    # 6. Customer complaint flag
    if 'NumberOfComplaints' in df.columns:
        df['has_complaint'] = (df['NumberOfComplaints'] > 0).astype(int)

    return df

df      = build_features(df)
holdout = build_features(holdout)
print("New shape:", df.shape)
print("New features:", [c for c in df.columns if c not in pd.read_csv(TRAIN_PATH).columns])

New shape: (51047, 60)
New features: ['drop_call_ratio', 'has_overage', 'tenure_band']


In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_holdout = holdout.drop(columns=[TARGET])
y_holdout  = holdout[TARGET]

print(f"X shape       : {X.shape}")
print(f"X_holdout     : {X_holdout.shape}")

X shape       : (51047, 59)
X_holdout     : (20000, 59)


In [9]:
# Stratified split preserves 28.8% churn ratio in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train : {X_train.shape}  churn: {round(y_train.mean(),3)}")
print(f"X_val   : {X_val.shape}    churn: {round(y_val.mean(),3)}")

X_train : (40837, 59)  churn: 0.288
X_val   : (10210, 59)    churn: 0.288


In [10]:
scaler = StandardScaler()

X_train_sc   = scaler.fit_transform(X_train)
X_val_sc     = scaler.transform(X_val)
X_holdout_sc = scaler.transform(X_holdout)

# Convert back to DataFrame (keeps column names)
X_train_sc   = pd.DataFrame(X_train_sc,   columns=X_train.columns)
X_val_sc     = pd.DataFrame(X_val_sc,     columns=X_val.columns)
X_holdout_sc = pd.DataFrame(X_holdout_sc, columns=X_holdout.columns)

print("Scaling done. Mean of X_train_sc:", X_train_sc.mean().mean().round(4))

Scaling done. Mean of X_train_sc: -0.0


In [11]:
# import os, joblib

# os.makedirs('../data/processed', exist_ok=True)
# os.makedirs('../data/features',  exist_ok=True)
# os.makedirs('../models',         exist_ok=True)

# # Save unscaled splits (for tree-based models like XGBoost)
# X_train.to_parquet('../data/processed/X_train.parquet')
# X_val.to_parquet('../data/processed/X_val.parquet')
# X_holdout.to_parquet('../data/processed/X_holdout.parquet')
# y_train.to_frame().to_parquet('../data/processed/y_train.parquet')
# y_val.to_frame().to_parquet('../data/processed/y_val.parquet')
# y_holdout.to_frame().to_parquet('../data/processed/y_holdout.parquet')

# # Save scaled splits (for Logistic Regression)
# X_train_sc.to_parquet('../data/features/X_train_scaled.parquet')
# X_val_sc.to_parquet('../data/features/X_val_scaled.parquet')
# X_holdout_sc.to_parquet('../data/features/X_holdout_scaled.parquet')

# # Save the scaler (needed by API at inference time)
# joblib.dump(scaler, '../models/scaler.pkl')

# print("All files saved successfully!")